# 02 · Parquet → Iceberg (Nessie + MinIO)
Lee el parquet del bucket `raw` y lo guarda como tabla Iceberg usando el catálogo Nessie.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv('/home/jovyan/.env')

MINIO_ENDPOINT       = os.environ['MINIO_ENDPOINT']
MINIO_ACCESS_KEY     = os.environ['MINIO_ACCESS_KEY']
MINIO_SECRET_KEY     = os.environ['MINIO_SECRET_KEY']
MINIO_BUCKET_RAW     = os.environ.get('MINIO_BUCKET_RAW',     'raw')
MINIO_BUCKET_ICEBERG = os.environ.get('MINIO_BUCKET_ICEBERG', 'iceberg')
MINIO_OBJECT_KEY     = os.environ.get('MINIO_OBJECT_KEY',     'nyc/yellow_tripdata_2025-01.parquet')
NESSIE_URL           = os.environ['NESSIE_URL']   # http://fhbd-nessie:19120/iceberg
NESSIE_BRANCH        = os.environ.get('NESSIE_BRANCH', 'main')
ICEBERG_NAMESPACE    = os.environ.get('ICEBERG_NAMESPACE', 'nyc')
ICEBERG_TABLE        = os.environ.get('ICEBERG_TABLE',     'yellow_tripdata')

LOCAL_PATH = '/home/jovyan/data/yellow_tripdata_2025-01.parquet'

print(' Variables cargadas')
print(f'  MinIO   : {MINIO_ENDPOINT}  bucket_raw={MINIO_BUCKET_RAW}  bucket_iceberg={MINIO_BUCKET_ICEBERG}')
print(f'  Nessie  : {NESSIE_URL}  branch={NESSIE_BRANCH}')
print(f'  Tabla   : {ICEBERG_NAMESPACE}.{ICEBERG_TABLE}')

 Variables cargadas
  MinIO   : http://fhbd-minio:9000  bucket_raw=raw  bucket_iceberg=iceberg
  Nessie  : http://fhbd-nessie:19120/iceberg  branch=main
  Tabla   : nyc.yellow_tripdata


## 1 · Asegurar bucket `iceberg` en MinIO

In [2]:
import boto3
from botocore.client import Config
from botocore.exceptions import ClientError

s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1',
)

def ensure_bucket(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        print(f'  Bucket "{bucket}" ya existe.')
    except ClientError:
        s3.create_bucket(Bucket=bucket)
        print(f'  Bucket "{bucket}" creado ')

ensure_bucket(MINIO_BUCKET_ICEBERG)

  Bucket "iceberg" ya existe.


## 2 · Descargar parquet desde MinIO `raw`

In [3]:
import os

os.makedirs(os.path.dirname(LOCAL_PATH), exist_ok=True)

print(f'Descargando s3://{MINIO_BUCKET_RAW}/{MINIO_OBJECT_KEY} ...')
s3.download_file(MINIO_BUCKET_RAW, MINIO_OBJECT_KEY, LOCAL_PATH)
size_mb = os.path.getsize(LOCAL_PATH) / 1024**2
print(f'   {LOCAL_PATH}  ({size_mb:.1f} MB)')

Descargando s3://raw/nyc/yellow_tripdata_2025-01.parquet ...
   /home/jovyan/data/yellow_tripdata_2025-01.parquet  (56.4 MB)


## 3 · Leer parquet con PyArrow

In [4]:
import pyarrow.parquet as pq
import pyarrow as pa

arrow_table = pq.read_table(LOCAL_PATH)

print(f'Filas     : {len(arrow_table):,}')
print(f'Columnas  : {len(arrow_table.schema)}')
print(f'\nSchema:\n{arrow_table.schema}')

Filas     : 3,475,226
Columnas  : 20

Schema:
VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double
cbd_congestion_fee: double


In [5]:
# Vista previa
arrow_table.slice(0, 5).to_pandas()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1,1.60,1,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1,0.50,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1,0.60,1,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


## 4 · Conectar al catálogo Nessie

In [6]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog(
    'nessie',
    **{
        'type': 'rest',
        'uri': NESSIE_URL,
        'ref': NESSIE_BRANCH,
        's3.endpoint': MINIO_ENDPOINT,
        's3.access-key-id': MINIO_ACCESS_KEY,
        's3.secret-access-key': MINIO_SECRET_KEY,
        's3.path-style-access': 'true',
        's3.region': 'us-east-1',
    },
)

print(f' Catálogo conectado → {NESSIE_URL}')

 Catálogo conectado → http://fhbd-nessie:19120/iceberg


## 5 · Crear namespace si no existe

In [7]:
ns = (ICEBERG_NAMESPACE,)

if ns not in catalog.list_namespaces():
    catalog.create_namespace(ns)
    print(f'Namespace "{ICEBERG_NAMESPACE}" creado ')
else:
    print(f'Namespace "{ICEBERG_NAMESPACE}" ya existe.')

print('Namespaces disponibles:', catalog.list_namespaces())

Namespace "nyc" creado 
Namespaces disponibles: [('nyc',)]


## 6 · Crear / sobreescribir tabla Iceberg

In [8]:
table_id = (ICEBERG_NAMESPACE, ICEBERG_TABLE)

if catalog.table_exists(table_id):
    print(f'Tabla "{".".join(table_id)}" ya existe — sobreescribiendo...')
    ice_table = catalog.load_table(table_id)
    ice_table.overwrite(arrow_table)
else:
    print(f'Creando tabla "{".".join(table_id)}"...')
    ice_table = catalog.create_table(
        table_id,
        schema=arrow_table.schema,
    )
    ice_table.append(arrow_table)

print(f'\n Tabla Iceberg guardada')
print(f'   Location: {ice_table.metadata_location}')

Creando tabla "nyc.yellow_tripdata"...

 Tabla Iceberg guardada
   Location: s3://iceberg/nyc/yellow_tripdata_c575d4ed-6257-4caa-b3c0-8b50fd687bec/metadata/00000-d2d3885f-49b3-4969-b453-d50a1609146d.metadata.json


## 7 · Verificación

In [9]:
# Leer de vuelta desde Iceberg
sample = ice_table.scan(limit=5).to_arrow()
print(f'Filas en tabla Iceberg (scan completo): {ice_table.scan().to_arrow().num_rows:,}')
print('\nPrimeras 5 filas:')
sample.to_pandas()

Filas en tabla Iceberg (scan completo): 3,475,226

Primeras 5 filas:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1,1.60,1,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1,0.50,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1,0.60,1,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0
